In [ ]:
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

# CNN feature extraction

In [ ]:
device = 'cuda' if torch.cuda.is_available else 'cpu'

# SVM classifier

In [208]:
data = pd.read_csv('./data/dataset_w_features.csv', index_col=0)

In [209]:
train = data[data['data_set'] == 'train']
test = data[data['data_set'] == 'test']
train.head()

,path,classification,medical_name,data_set,diameter,area,perimeter,circularity,saturation_std,vue_std,...,p16,p17,p18,p19,p20,p21,p22,p23,p24,p25
0,./ISIC/Train/pigmented benign keratosis/ISIC_0...,benignant,pigmented benign keratosis,train,163.0,14789.0,478.558436,0.811482,25.701140,23.954772,...,351.0,294.0,272.0,260.0,273.0,300.0,286.0,298.0,721.0,6628.0
1,./ISIC/Train/vascular lesion/ISIC_0026693.jpg,benignant,vascular lesion,train,91.0,4074.0,250.936073,0.813026,33.442159,6.723879,...,146.0,108.0,83.0,78.0,55.0,60.0,52.0,58.0,165.0,1166.0
2,./ISIC/Train/actinic keratosis/ISIC_0027334.jpg,malignant,actinic keratosis,train,204.0,27512.0,630.499564,0.869686,18.707230,10.340140,...,699.0,629.0,527.0,482.0,497.0,454.0,461.0,431.0,1366.0,9288.0
3,./ISIC/Test/vascular lesion/ISIC_0024375.jpg,benignant,vascular lesion,train,40.0,970.0,117.740115,0.879291,31.692214,7.582935,...,54.0,33.0,21.0,21.0,14.0,14.0,15.0,15.0,35.0,286.0
4,./ISIC/Test/squamous cell carcinoma/ISIC_00245...,malignant,squamous cell carcinoma,train,207.0,23095.5,645.452879,0.696640,22.041543,9.975496,...,587.0,509.0,430.0,406.0,352.0,350.0,432.0,456.0,1004.0,9418.0


In [210]:
X_train = train[['diameter','perimeter', 'circularity', 'saturation_std', 'vue_std'] + [f'p{i}' for i in range(26)]]
X_test = test[['diameter', 'perimeter', 'circularity', 'saturation_std', 'vue_std'] + [f'p{i}' for i in range(26)]]

y_train = train[['classification']]
y_test = test[['classification']]

In [211]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [212]:
label_encoder = LabelEncoder()
label_encoder.fit(y_train)
y_train_encoded = label_encoder.transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_label.py:93: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


In [213]:
parameters = {'kernel': ('linear', 'rbf'), 'C': [1,10, 100], 'gamma': ['scale', 'auto']}
model_svm = svm.SVC()
gridSe = GridSearchCV(model_svm, parameters)
gridSe.fit(X_train_scaled, y_train_encoded)
model_svm = gridSe.best_estimator_


In [214]:
y_pred_svm = model_svm.predict(X_test_scaled)

In [215]:
print(classification_report(y_test_encoded, y_pred_svm))

              precision    recall  f1-score   support

           0       0.67      0.58      0.62       237
           1       0.62      0.71      0.66       235

    accuracy                           0.64       472
   macro avg       0.64      0.64      0.64       472
weighted avg       0.64      0.64      0.64       472

